# ExamGuard v0.1 — All Detectors Combined

Head + Eyes + Body + Talking = Full cheating detection.

Run cells 1-4 in order. Press 'q' to quit.

In [1]:
# CELL 1: Load models
import cv2, time, numpy as np, mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from collections import deque

face_landmarker = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(
        model_asset_path='../01_head_direction/face_landmarker.task'),
    num_faces=1, min_face_detection_confidence=0.5, min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5, running_mode=vision.RunningMode.IMAGE))

pose_landmarker = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(
        model_asset_path='../03_body_turning/pose_landmarker.task'),
    num_poses=1, min_pose_detection_confidence=0.5, min_tracking_confidence=0.5,
    running_mode=vision.RunningMode.IMAGE))

print('Both models loaded!')

Both models loaded!


In [2]:
# CELL 2: Configuration — ALL detectors

# HEAD
HEAD_TURN_THRESHOLD = 0.15
HEAD_UP_THRESHOLD = 0.1
HEAD_WEIGHT = 35

# EYES
GAZE_THRESHOLD = 0.12
EYE_WEIGHT = 30
HISTORY_SIZE = 300
MIN_READINGS = 30

# BODY
BODY_TURN_THRESHOLD = 0.06
BODY_WEIGHT = 15

# TALKING
MOUTH_OPEN_THRESHOLD = 0.15
TALKING_CYCLES_NEEDED = 3
CYCLE_WINDOW = 2.0
TALKING_WEIGHT = 20

# ALERT LEVELS
MILD_THRESHOLD = 20
SUSPICIOUS_THRESHOLD = 40
HIGH_THRESHOLD = 65

print('ExamGuard v0.1 Configuration')
print(f'  Head:{HEAD_WEIGHT}  Eyes:{EYE_WEIGHT}  Body:{BODY_WEIGHT}  Talking:{TALKING_WEIGHT}  = {HEAD_WEIGHT+EYE_WEIGHT+BODY_WEIGHT+TALKING_WEIGHT} max')
print(f'  Levels: Clear<{MILD_THRESHOLD}, Mild<{SUSPICIOUS_THRESHOLD}, Suspicious<{HIGH_THRESHOLD}, High>={HIGH_THRESHOLD}')

ExamGuard v0.1 Configuration
  Head:35  Eyes:30  Body:15  Talking:20  = 100 max
  Levels: Clear<20, Mild<40, Suspicious<65, High>=65


In [3]:
# CELL 3: All detection functions

def detect_head(fl, w, h):
    nose_x = fl[4].x * w
    eye_cx = (fl[133].x * w + fl[362].x * w) / 2
    eye_cy = (fl[133].y + fl[362].y) / 2 * h
    eye_dist = abs(fl[362].x * w - fl[133].x * w)
    if eye_dist == 0: return 'FORWARD', 0
    hr = (nose_x - eye_cx) / eye_dist
    vr = (fl[4].y * h - eye_cy) / eye_dist
    if vr < -HEAD_UP_THRESHOLD: d = 'UP'
    elif hr < -HEAD_TURN_THRESHOLD: d = 'LEFT'
    elif hr > HEAD_TURN_THRESHOLD: d = 'RIGHT'
    else: return 'FORWARD', 0
    return d, min(HEAD_WEIGHT, int((max(abs(hr), abs(vr)) / 0.4) * HEAD_WEIGHT))

def detect_eyes(fl, w, h, center):
    lw = abs(fl[133].x - fl[33].x); rw = abs(fl[362].x - fl[263].x)
    lr = (fl[468].x - fl[33].x) / lw if lw > 0 else 0.5
    rr = (fl[473].x - fl[263].x) / rw if rw > 0 else 0.5
    avg = (lr + rr) / 2; dev = avg - center
    if abs(dev) < GAZE_THRESHOLD: return 'CENTER', 0, avg
    d = 'LEFT' if dev > 0 else 'RIGHT'
    return d, min(EYE_WEIGHT, int((abs(dev) / 0.4) * EYE_WEIGHT)), avg

def detect_body(pl, w, h):
    yd = pl[12].y - pl[11].y
    vd = 0
    if hasattr(pl[11], 'visibility') and hasattr(pl[12], 'visibility'):
        vd = pl[11].visibility - pl[12].visibility
    rot = abs(yd) + abs(vd) * 0.3
    if rot < BODY_TURN_THRESHOLD: return 'FORWARD', 0
    d = 'LEFT' if yd > 0 else 'RIGHT'
    return d, min(BODY_WEIGHT, int((rot / 0.15) * BODY_WEIGHT))

class TalkingDetector:
    def __init__(self):
        self.prev_open = False
        self.cycle_times = deque()
    
    def update(self, fl, w, h, now):
        # Mouth aspect ratio
        vert = abs(fl[13].y - fl[14].y) * h
        horiz = abs(fl[308].x - fl[78].x) * w
        mar = vert / horiz if horiz > 0 else 0
        
        is_open = mar > MOUTH_OPEN_THRESHOLD
        if self.prev_open and not is_open:
            self.cycle_times.append(now)
        self.prev_open = is_open
        
        while self.cycle_times and (now - self.cycle_times[0]) > CYCLE_WINDOW:
            self.cycle_times.popleft()
        
        cycles = len(self.cycle_times)
        talking = cycles >= TALKING_CYCLES_NEEDED
        score = min(TALKING_WEIGHT, int((cycles / 8) * TALKING_WEIGHT)) if talking else 0
        return talking, cycles, score, mar

print('All 4 detectors ready!')

All 4 detectors ready!


In [ ]:
# CELL 4: ExamGuard v0.1 — LIVE
cam = cv2.VideoCapture(0)
cam.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cam.isOpened():
    print('Cannot open webcam!')
else:
    print('='*45)
    print('  EXAMGUARD v0.1 — RUNNING')
    print('  Head + Eyes + Body + Talking')
    print('  Press q to quit')
    print('='*45)
    
    gh = []; ec = 0.5
    talker = TalkingDetector()
    fc = 0; t0 = time.time()
    alert_log = []  # Store all alerts with timestamps
    
    while True:
        ok, f = cam.read()
        if not ok: break
        f = cv2.flip(f, 1); fc += 1
        h, w = f.shape[:2]
        now = time.time()
        
        rgb = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        mi = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        face_r = face_landmarker.detect(mi)
        pose_r = pose_landmarker.detect(mi)
        
        hs=0; es=0; bs=0; ts=0
        hd='--'; ed='--'; bd='--'; td='Silent'
        mar_val = 0
        
        # === FACE (head + eyes + talking) ===
        if face_r.face_landmarks and len(face_r.face_landmarks) > 0:
            fl = face_r.face_landmarks[0]
            
            # Head
            hd, hs = detect_head(fl, w, h)
            
            # Eyes
            ed, es, avg = detect_eyes(fl, w, h, ec)
            gh.append(avg)
            if len(gh) > HISTORY_SIZE: gh.pop(0)
            if len(gh) >= MIN_READINGS:
                ec = np.median(gh)
            else:
                ed = f'wait({len(gh)}/{MIN_READINGS})'; es = 0
            
            # Talking
            is_talk, cycles, ts, mar_val = talker.update(fl, w, h, now)
            td = f'TALKING({cycles})' if is_talk else f'Silent({cycles})'
            
            # Draw face points
            for i in [4,133,362]: cv2.circle(f,(int(fl[i].x*w),int(fl[i].y*h)),2,(0,255,255),-1)
            for i in [468,473]: cv2.circle(f,(int(fl[i].x*w),int(fl[i].y*h)),3,(255,255,0),-1)
            # Mouth outline
            mouth = [78,191,80,81,82,13,312,311,310,415,308,324,318,402,317,14,87,178,88,95]
            for i in range(len(mouth)-1):
                p1=(int(fl[mouth[i]].x*w),int(fl[mouth[i]].y*h))
                p2=(int(fl[mouth[i+1]].x*w),int(fl[mouth[i+1]].y*h))
                mc = (0,0,255) if is_talk else (0,200,0)
                cv2.line(f,p1,p2,mc,1)
        
        # === POSE (body) ===
        if pose_r.pose_landmarks and len(pose_r.pose_landmarks) > 0:
            pl = pose_r.pose_landmarks[0]
            bd, bs = detect_body(pl, w, h)
            for i in [11,12]: cv2.circle(f,(int(pl[i].x*w),int(pl[i].y*h)),5,(0,165,255),-1)
            cv2.line(f,(int(pl[11].x*w),int(pl[11].y*h)),(int(pl[12].x*w),int(pl[12].y*h)),(0,165,255),2)
        
        # === COMBINED SCORE ===
        total = min(100, hs + es + bs + ts)
        
        # Color helper
        def sc(s, mx):
            if s == 0: return (0,180,0)
            elif s < mx * 0.5: return (0,180,200)
            else: return (0,0,200)
        
        # === 4 SIGNAL BARS AT TOP ===
        signals = [
            (f'Head:{hd}', hs, HEAD_WEIGHT),
            (f'Eyes:{ed}', es, EYE_WEIGHT),
            (f'Body:{bd}', bs, BODY_WEIGHT),
            (f'Mouth:{td}', ts, TALKING_WEIGHT),
        ]
        for i, (txt, s, mx) in enumerate(signals):
            y = i * 24
            c = sc(s, mx)
            cv2.rectangle(f, (0, y), (w, y + 22), c, -1)
            cv2.putText(f, f'{txt} [{s}/{mx}]', (10, y + 16),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        # === SCORE BAR AT BOTTOM ===
        cv2.rectangle(f, (0, h-55), (w, h), (25, 25, 25), -1)
        bw = int((total / 100) * (w - 20))
        
        if total < MILD_THRESHOLD:
            bc = (0,200,0); v = 'ALL CLEAR'
        elif total < SUSPICIOUS_THRESHOLD:
            bc = (0,200,200); v = 'MILD WARNING'
        elif total < HIGH_THRESHOLD:
            bc = (0,100,255); v = 'SUSPICIOUS!'
        else:
            bc = (0,0,255); v = 'HIGH ALERT!'
        
        # Score bar
        cv2.rectangle(f, (10, h-50), (10+bw, h-32), bc, -1)
        cv2.rectangle(f, (10, h-50), (w-10, h-32), (80,80,80), 1)
        
        # Verdict text
        cv2.putText(f, f'{v} | Score:{total}/100', (10, h-8),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.65, bc, 2)
        
        # Log alerts
        if total >= SUSPICIOUS_THRESHOLD:
            elapsed = now - t0
            alert_log.append({
                'time': elapsed,
                'score': total,
                'head': hd,
                'eyes': ed,
                'body': bd,
                'talking': td,
                'verdict': v
            })
        
        # FPS
        elapsed = now - t0
        fps = fc / elapsed if elapsed > 0 else 0
        cv2.putText(f, f'FPS:{fps:.0f}', (w-70, h-8),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, (120,120,120), 1)
        
        cv2.imshow('ExamGuard v0.1 (q=quit)', f)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
    
    cam.release()
    cv2.destroyAllWindows()
    
    # Session report
    print()
    print('='*45)
    print('  SESSION REPORT')
    print('='*45)
    print(f'  Duration: {elapsed:.0f} seconds')
    print(f'  FPS: {fps:.0f}')
    print(f'  Total alerts: {len(alert_log)}')
    if alert_log:
        print()
        print('  Alert Timeline:')
        seen_times = set()
        for a in alert_log:
            t_key = int(a["time"])
            if t_key not in seen_times:
                seen_times.add(t_key)
                mins = int(a["time"]) // 60
                secs = int(a["time"]) % 60
                print(f'    {mins:02d}:{secs:02d} — {a["verdict"]} (score:{a["score"]}) '
                      f'Head:{a["head"]} Eyes:{a["eyes"]} Body:{a["body"]} {a["talking"]}')
    print('='*45)

  EXAMGUARD v0.1 — RUNNING
  Head + Eyes + Body + Talking
  Press q to quit


## ExamGuard v0.1 COMPLETE!

### What you built:
- Head direction detection (left/right/up)
- Eye gaze tracking (peek detection even with head forward)
- Body rotation detection (shoulder angle)
- Talking/whispering detection (mouth cycle counting)
- Combined suspicion score (0-100)
- 4 alert levels (clear/mild/suspicious/high)
- Session report with alert timeline
- Auto-calibration (no setup needed)

### Next: Web App
Module 03 — Build a Flask web app where you can:
- Upload a recorded exam video → get cheating timeline
- Connect live webcam → real-time detection in browser